# 11 — Improved Unsupervised Clustering

Three targeted experiments on top of the notebook 02 baseline (MiniLM KMeans/Agglomerative ~45% ACC):

**Experiment A — MPNet embeddings:** replace `all-MiniLM-L6-v2` (384-dim) with
`all-mpnet-base-v2` (768-dim) on the same BART summaries. Run KMeans, Agglomerative, DEC.

**Experiment B — UMAP hyperparameter tuning:** keep MiniLM embeddings; sweep
n_neighbors=[15,30,50] × min_dist=[0.0,0.05,0.1] (9 combos), pick best by silhouette.
Run KMeans and Agglomerative with best params. (DEC has its own dim-reduction; excluded.)

**Experiment C — Title embeddings:** embed raw article titles with MiniLM instead of BART
summaries. Run KMeans, Agglomerative, DEC.

New methods (8 total): mpnet_kmeans, mpnet_agglomerative, mpnet_dec,
minilm_kmeans_umap, minilm_agglomerative_umap,
title_minilm_kmeans, title_minilm_agglomerative, title_minilm_dec

All methods produce the same artifact schema as notebook 02 and are automatically
included in the comparison table when notebook 07 is re-run.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

from utils import config
from utils.dec import train_dec
from utils.embeddings import load_cached
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised, hungarian_match_predictions
from utils.samples import save_full_output

# ── Data ─────────────────────────────────────────────────────────────────────
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
true_labels  = train_clean["label"].to_numpy()
texts        = train_clean["text"].tolist()
summaries    = train_clean["summary"].tolist()
titles       = train_clean["title"].tolist()

assert len(true_labels) == 3824, f"Expected 3824 rows, got {len(true_labels)}"
print(f"Rows: {len(train_clean)} | k={config.NUM_CLASSES}")

# ── Accumulator ───────────────────────────────────────────────────────────────
all_results = {}   # method_name -> metrics dict
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Rows: 3824 | k=16


In [2]:
def umap_reduce(
    emb: np.ndarray,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    n_components: int = 50,
    metric: str = "cosine",
    seed: int = config.SEED,
) -> np.ndarray:
    """Reduce embeddings via UMAP. Parameterised to support grid search."""
    reducer = umap.UMAP(
        n_components=n_components,
        metric=metric,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=seed,
    )
    return reducer.fit_transform(emb)


def run_and_save(name: str, cluster_labels: np.ndarray, emb_original: np.ndarray) -> None:
    """Evaluate and save all artifacts for one clustering result (same schema as notebook 02)."""
    metrics = evaluate_unsupervised(
        true_labels, cluster_labels, emb_original,
        metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED,
    )
    all_results[name] = metrics

    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    np.save(config.RESULTS_DIR / f"cluster_labels_{name}.npy", cluster_labels)

    interp = summarize_clusters(
        summaries, cluster_labels, true_labels,
        emb_original,
        config.CLASS_NAMES,
    )
    interp.to_csv(config.RESULTS_DIR / f"clusters_{name}.csv", index=False)

    predicted = hungarian_match_predictions(true_labels, cluster_labels)
    save_full_output(
        texts, predicted, true_labels, config.CLASS_NAMES,
        extra_columns={"summary": summaries},
        path=config.RESULTS_DIR / f"full_labels_{name}.csv",
    )

    print(f"  {name}: ACC={metrics['ACC (Hungarian)']:.4f}  "
          f"F1={metrics['Macro F1']:.4f}  "
          f"NMI={metrics['NMI']:.4f}  "
          f"coverage={metrics['Coverage']:.2f}")


print("Helpers defined: umap_reduce(), run_and_save()")

Helpers defined: umap_reduce(), run_and_save()


In [3]:
# ════════════════════════════════════════════════════════════════════
# EXPERIMENT A — MPNet embeddings (all-mpnet-base-v2, 768-dim)
# ════════════════════════════════════════════════════════════════════
MPNET_CACHE = REPO_ROOT / "embeddings_cache" / "mpnet_train_full_summary.npy"

if MPNET_CACHE.exists():
    print(f"Loading cached MPNet embeddings from {MPNET_CACHE.name}...")
    mpnet_emb = np.load(MPNET_CACHE).astype(np.float32)
else:
    print("Encoding summaries with all-mpnet-base-v2 (first run — takes ~5 min)...")
    model_mpnet = SentenceTransformer("all-mpnet-base-v2")
    mpnet_emb = model_mpnet.encode(
        summaries,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(MPNET_CACHE, mpnet_emb)
    print(f"Saved to {MPNET_CACHE.name}")

assert mpnet_emb.shape == (3824, 768), f"Unexpected shape: {mpnet_emb.shape}"
print(f"MPNet embeddings ready: {mpnet_emb.shape}")

Encoding summaries with all-mpnet-base-v2 (first run — takes ~5 min)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Saved to mpnet_train_full_summary.npy
MPNet embeddings ready: (3824, 768)


In [4]:
print("=== [A] MPNet KMeans ===")
mpnet_reduced = umap_reduce(mpnet_emb)   # default params: n_neighbors=15, min_dist=0.1
print(f"  UMAP reduced: {mpnet_reduced.shape}")

km = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
run_and_save("mpnet_kmeans", km.fit_predict(mpnet_reduced), mpnet_emb)

print("\n=== [A] MPNet Agglomerative ===")
agg = AgglomerativeClustering(
    n_clusters=config.NUM_CLASSES,
    linkage="ward",
    metric="euclidean",
)
run_and_save("mpnet_agglomerative", agg.fit_predict(mpnet_reduced), mpnet_emb)

=== [A] MPNet KMeans ===


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP reduced: (3824, 50)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  mpnet_kmeans: ACC=0.5413  F1=0.5379  NMI=0.4468  coverage=1.00

=== [A] MPNet Agglomerative ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  mpnet_agglomerative: ACC=0.4882  F1=0.4764  NMI=0.4363  coverage=1.00


In [ ]:
print("\n=== [A] MPNet DEC ===")
print(f"Training DEC on mpnet embeddings {mpnet_emb.shape} — expect ~10-15 min on CPU...")
labels_dec, _, _ = train_dec(
    mpnet_emb,
    n_clusters=config.NUM_CLASSES,
    latent_dim=64,
    pretrain_epochs=50,
    dec_epochs=150,
    batch_size=256,
    lr_pretrain=1e-3,
    lr_dec=1e-4,
    update_interval=5,
    tol=1e-3,
    seed=config.SEED,
)
run_and_save("mpnet_dec", labels_dec, mpnet_emb)

In [6]:
# ════════════════════════════════════════════════════════════════════
# EXPERIMENT B — UMAP hyperparameter tuning on MiniLM embeddings
# ════════════════════════════════════════════════════════════════════
minilm_emb = np.load(
    REPO_ROOT / "embeddings_cache" / "minilm_train_full_summary.npy"
).astype(np.float32)
assert minilm_emb.shape == (3824, 384), f"Unexpected: {minilm_emb.shape}"
print(f"MiniLM embeddings loaded: {minilm_emb.shape}")

N_NEIGHBORS_GRID = [15, 30, 50]
MIN_DIST_GRID    = [0.0, 0.05, 0.1]

print("\nUMAP grid search (3x3 = 9 combos, scored by KMeans silhouette):")
print(f"{'n_neighbors':>12} {'min_dist':>9} {'silhouette':>12}")
print("-" * 36)

best_score, best_n_neighbors, best_min_dist = -1.0, 15, 0.1

for nn in N_NEIGHBORS_GRID:
    for md in MIN_DIST_GRID:
        reduced = umap_reduce(minilm_emb, n_neighbors=nn, min_dist=md)
        km_tmp = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
        labels_tmp = km_tmp.fit_predict(reduced)
        score = silhouette_score(
            reduced, labels_tmp,
            sample_size=config.SILHOUETTE_SAMPLE_SIZE,
            random_state=config.SEED,
        )
        marker = " <- best" if score > best_score else ""
        print(f"{nn:>12} {md:>9.2f} {score:>12.4f}{marker}")
        if score > best_score:
            best_score, best_n_neighbors, best_min_dist = score, nn, md

print(f"\nBest params: n_neighbors={best_n_neighbors}, min_dist={best_min_dist} "
      f"(silhouette={best_score:.4f})")

MiniLM embeddings loaded: (3824, 384)

UMAP grid search (3x3 = 9 combos, scored by KMeans silhouette):
 n_neighbors  min_dist   silhouette
------------------------------------


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          15      0.00       0.4234 <- best


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          15      0.05       0.3911


...

          50      0.00       0.3986


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          50      0.05       0.3865


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          50      0.10       0.3511

Best params: n_neighbors=15, min_dist=0.0 (silhouette=0.4234)


In [7]:
print("\n=== [B] MiniLM KMeans (tuned UMAP) ===")
minilm_reduced_tuned = umap_reduce(
    minilm_emb, n_neighbors=best_n_neighbors, min_dist=best_min_dist
)
print(f"  UMAP reduced: {minilm_reduced_tuned.shape}")

km = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
run_and_save("minilm_kmeans_umap", km.fit_predict(minilm_reduced_tuned), minilm_emb)

print("\n=== [B] MiniLM Agglomerative (tuned UMAP) ===")
agg = AgglomerativeClustering(
    n_clusters=config.NUM_CLASSES,
    linkage="ward",
    metric="euclidean",
)
run_and_save("minilm_agglomerative_umap", agg.fit_predict(minilm_reduced_tuned), minilm_emb)


=== [B] MiniLM KMeans (tuned UMAP) ===


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP reduced: (3824, 50)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  minilm_kmeans_umap: ACC=0.4746  F1=0.4695  NMI=0.4188  coverage=1.00

=== [B] MiniLM Agglomerative (tuned UMAP) ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  minilm_agglomerative_umap: ACC=0.4772  F1=0.4666  NMI=0.4020  coverage=1.00


In [8]:
# ════════════════════════════════════════════════════════════════════
# EXPERIMENT C — Title embeddings (MiniLM on raw article titles)
# ════════════════════════════════════════════════════════════════════
TITLE_CACHE = REPO_ROOT / "embeddings_cache" / "minilm_train_title.npy"

if TITLE_CACHE.exists():
    print(f"Loading cached title embeddings from {TITLE_CACHE.name}...")
    title_emb = np.load(TITLE_CACHE).astype(np.float32)
else:
    print("Encoding titles with all-MiniLM-L6-v2 (first run — takes ~1 min)...")
    model_minilm = SentenceTransformer("all-MiniLM-L6-v2")
    title_emb = model_minilm.encode(
        titles,
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(TITLE_CACHE, title_emb)
    print(f"Saved to {TITLE_CACHE.name}")

assert title_emb.shape == (3824, 384), f"Unexpected shape: {title_emb.shape}"
print(f"Title embeddings ready: {title_emb.shape}")

Encoding titles with all-MiniLM-L6-v2 (first run — takes ~1 min)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Saved to minilm_train_title.npy
Title embeddings ready: (3824, 384)


In [9]:
print("=== [C] Title MiniLM KMeans ===")
title_reduced = umap_reduce(title_emb)   # default params
print(f"  UMAP reduced: {title_reduced.shape}")

km = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
run_and_save("title_minilm_kmeans", km.fit_predict(title_reduced), title_emb)

print("\n=== [C] Title MiniLM Agglomerative ===")
agg = AgglomerativeClustering(
    n_clusters=config.NUM_CLASSES,
    linkage="ward",
    metric="euclidean",
)
run_and_save("title_minilm_agglomerative", agg.fit_predict(title_reduced), title_emb)

=== [C] Title MiniLM KMeans ===


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP reduced: (3824, 50)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  title_minilm_kmeans: ACC=0.5154  F1=0.5132  NMI=0.4433  coverage=1.00

=== [C] Title MiniLM Agglomerative ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  title_minilm_agglomerative: ACC=0.5207  F1=0.5202  NMI=0.4352  coverage=1.00


In [ ]:
print("\n=== [C] Title MiniLM DEC ===")
print(f"Training DEC on title embeddings {title_emb.shape} — expect ~10-15 min on CPU...")
labels_title_dec, _, _ = train_dec(
    title_emb,
    n_clusters=config.NUM_CLASSES,
    latent_dim=64,
    pretrain_epochs=50,
    dec_epochs=150,
    batch_size=256,
    lr_pretrain=1e-3,
    lr_dec=1e-4,
    update_interval=5,
    tol=1e-3,
    seed=config.SEED,
)
run_and_save("title_minilm_dec", labels_title_dec, title_emb)

In [11]:
# ════════════════════════════════════════════════════════════════════
# RESULTS SUMMARY — all 8 new methods vs. baseline
# ════════════════════════════════════════════════════════════════════
ALL_NEW_METHODS = [
    "mpnet_kmeans", "mpnet_agglomerative", "mpnet_dec",
    "minilm_kmeans_umap", "minilm_agglomerative_umap",
    "title_minilm_kmeans", "title_minilm_agglomerative", "title_minilm_dec",
]

assert set(ALL_NEW_METHODS) == set(all_results.keys()), (
    f"Missing methods: {set(ALL_NEW_METHODS) - set(all_results.keys())}"
)

print(f"{'Method':<35} {'ACC':>7} {'F1':>7} {'NMI':>7} {'Silhouette':>11}")
print("-" * 70)
print(f"{'minilm_agglomerative (baseline)':<35} {'0.4506':>7}   -       -        -")
for name in ALL_NEW_METHODS:
    m = all_results[name]
    beat = " ✓" if m["ACC (Hungarian)"] > 0.4506 else ""
    print(
        f"{name:<35} {m['ACC (Hungarian)']:>7.4f} "
        f"{m['Macro F1']:>7.4f} {m['NMI']:>7.4f} "
        f"{m['Silhouette Score']:>11.4f}{beat}"
    )
print("\n✓ = beats baseline (minilm_agglomerative ACC=0.4506)")
print(f"\nAll {len(ALL_NEW_METHODS)} new methods complete.")

Method                                  ACC      F1     NMI  Silhouette
----------------------------------------------------------------------
minilm_agglomerative (baseline)      0.4506   -       -        -
mpnet_kmeans                         0.5413  0.5379  0.4468      0.0408 ✓
mpnet_agglomerative                  0.4882  0.4764  0.4363      0.0416 ✓
mpnet_dec                            0.2670  0.2415  0.2595     -0.0079
minilm_kmeans_umap                   0.4746  0.4695  0.4188      0.0392 ✓
minilm_agglomerative_umap            0.4772  0.4666  0.4020      0.0337 ✓
title_minilm_kmeans                  0.5154  0.5132  0.4433      0.0372 ✓
title_minilm_agglomerative           0.5207  0.5202  0.4352      0.0319 ✓
title_minilm_dec                     0.2814  0.2879  0.2584      0.0009

✓ = beats baseline (minilm_agglomerative ACC=0.4506)

All 8 new methods complete.
